# Uso de `NLITextPreprocesser`

Este notebook carga `data/train.csv`, muestra los resultados de los métodos principales de `preprocesser.py` y exporta el dataset preprocesado a `data/train_preprocessed.csv`.

In [35]:
import nltk

nltk.download("punkt_tab")
nltk.download("wordnet")
nltk.download("stopwords")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/alballr/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /home/alballr/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/alballr/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [26]:
from pathlib import Path

import pandas as pd

from preprocesser import NLITextPreprocesser

columnas = ["id", "premise", "hypothesis", "lang_abv", "language", "label"]

DATA_PATH = Path("data/train.csv")
OUTPUT_PATH = Path("data/train_preprocessed.csv")

df_train = pd.read_csv(DATA_PATH)

df_train["label"] = pd.to_numeric(df_train["label"]).astype(int)

result_dict = {0: "entailment", 1: "neutral", 2: "contradiction"}

print(f"Archivo: {DATA_PATH}")
print(f"Filas totales: {len(df_train)}")

display(df_train.head())


Archivo: data/train.csv
Filas totales: 12120


,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
2,3931fbe82a,Des petites choses comme celles-là font une di...,J'essayais d'accomplir quelque chose.,fr,French,0
3,5622f0c60b,you know they can't really defend themselves l...,They can't defend themselves because of their ...,en,English,0
4,86aaa48b45,ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...,เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร,th,Thai,1


In [50]:
preprocessor = NLITextPreprocesser(
    columns=("premise", "hypothesis"),
    language="en",
    language_column="lang_abv",
    lowercase=True,
    strip_accents=True,
    tokenizer="nltk",
    stopword_method="nltk",
    keep_negations=True,
    custom_stopwords=("said", "issues"),
    reduction="lancaster",
    ngram_orders=(2, 3),
    top_n=10,
)
print("Configuración:")
print(pd.Series(preprocessor.get_config()).to_string())

Configuración:
columns             [premise, hypothesis]
language                               en
language_column                  lang_abv
lowercase                            True
strip_accents                        True
tokenizer                            nltk
stopword_method                      nltk
keep_negations                       True
custom_stopwords           [issues, said]
reduction                       lancaster
ngram_orders                       [2, 3]
top_n                                  10


In [51]:
example_texts = df_train["premise"].dropna().head(5).tolist()
example_text = example_texts[0]

print("Texto original:", example_text)
print("Texto estandarizado:", preprocessor.standardize_text(example_text))
print("Tokens (whitespace):", preprocessor.tokenize_text(example_text, "whitespace"))
print("Tokens (spaCy):", preprocessor.tokenize_text(example_text, "spacy"))

tokenizer_comparison = preprocessor.compare_tokenizers(
    example_texts, methods=("whitespace", "spacy")
)
print("\nComparación de tokenizadores:")
print(tokenizer_comparison.to_string(index=False))

Texto original: and these comments were considered in formulating the interim rules.
Texto estandarizado: and these comments were considered in formulating the interim rules.
Tokens (whitespace): ['and', 'these', 'comments', 'were', 'considered', 'in', 'formulating', 'the', 'interim', 'rules.']
Tokens (spaCy): ['and', 'these', 'comments', 'were', 'considered', 'in', 'formulating', 'the', 'interim', 'rules']

Comparación de tokenizadores:
    method  mean_tokens_per_text  vocabulary_size                                                                   sample_tokens
whitespace                  12.6               59 [and, these, comments, were, considered, in, formulating, the, interim, rules.]
     spacy                  13.0               61  [and, these, comments, were, considered, in, formulating, the, interim, rules]


In [52]:
print("Stopwords (ninguna):", len(preprocessor.get_stopwords("none")))
print("Primeras 15 stopwords de scikit-learn:", sorted(preprocessor.get_stopwords("sklearn"))[:15])
print("Stopwords personalizadas:", sorted(preprocessor.get_stopwords("custom")))

sample_tokens = preprocessor.tokenize_text("This is not a useful example, but it works.")
print("\nTokens de ejemplo:", sample_tokens)
print("Tokens sin stopwords:", preprocessor.remove_stopwords(sample_tokens))
print("Bigramas:", preprocessor.make_ngrams(sample_tokens, 2))

Stopwords (ninguna): 0
Primeras 15 stopwords de scikit-learn: ['a', 'about', 'above', 'across', 'after', 'afterwards', 'again', 'against', 'all', 'almost', 'alone', 'along', 'already', 'also', 'although']
Stopwords personalizadas: ['issues', 'said']

Tokens de ejemplo: ['this', 'is', 'not', 'a', 'useful', 'example', 'but', 'it', 'works']
Tokens sin stopwords: ['not', 'useful', 'example', 'works']
Bigramas: ['this is', 'is not', 'not a', 'a useful', 'useful example', 'example but', 'but it', 'it works']


In [53]:
tokenized_examples = [preprocessor.tokenize_text(text) for text in example_texts]
stopword_summary, stopword_tables = preprocessor.compare_stopword_configs(tokenized_examples, methods=("none", "sklearn", "custom"), top_n=5)

print("Resumen de configuraciones de stopwords:")
print(stopword_summary.to_string(index=False))
for method, table in stopword_tables.items():
    print(f"\nTokens principales con stopwords='{method}':")
    print(table.to_string(index=False))

Resumen de configuraciones de stopwords:
 method  token_count  vocabulary_size
   none           61               57
sklearn           40               39
 custom           59               55

Tokens principales con stopwords='none':
token  count  token_percentage  text_frequency  text_presence_freq
these      2          0.032787               2                 0.4
   in      2          0.032787               2                 0.4
  you      2          0.032787               1                 0.2
 know      2          0.032787               1                 0.2
  and      1          0.016393               1                 0.2

Tokens principales con stopwords='sklearn':
      token  count  token_percentage  text_frequency  text_presence_freq
       know      2             0.050               1                 0.2
   comments      1             0.025               1                 0.2
 considered      1             0.025               1                 0.2
formulating      1        

In [54]:
print("Tokens sin reducción:", preprocessor.reduce_tokens(sample_tokens, "none"))
print("Tokens con Porter:", preprocessor.reduce_tokens(sample_tokens, "porter"))

reduction_methods = ("none", "porter", "lancaster", "snowball", "spacy_lemma")
reduction_summary, reduction_tables = preprocessor.compare_reductions(
    tokenized_examples,
    methods=reduction_methods,
    top_n=5,
)
print("\nResumen de métodos de reducción:")
print(reduction_summary.to_string(index=False))
for method, table in reduction_tables.items():
    print(f"\nTokens principales con reducción='{method}':")
    print(table.to_string(index=False))

Tokens sin reducción: ['this', 'is', 'not', 'a', 'useful', 'example', 'but', 'it', 'works']
Tokens con Porter: ['thi', 'is', 'not', 'a', 'use', 'exampl', 'but', 'it', 'work']

Resumen de métodos de reducción:
     method  vocabulary_size  token_count
       none               57           61
     porter               56           61
  lancaster               56           61
   snowball               57           61
spacy_lemma               55           61

Tokens principales con reducción='none':
token  count  token_percentage  text_frequency  text_presence_freq
these      2          0.032787               2                 0.4
   in      2          0.032787               2                 0.4
  you      2          0.032787               1                 0.2
 know      2          0.032787               1                 0.2
  and      1          0.016393               1                 0.2

Tokens principales con reducción='porter':
token  count  token_percentage  text_frequency  tex

In [55]:
english_rows = df_train["lang_abv"].eq(preprocessor.language).sum()
print(f"Filas en inglés: {english_rows}")

preprocessed_df = preprocessor.preprocess_dataset(df_train)
print(f"\nDataset preprocesado: {preprocessed_df.shape[0]} filas x {preprocessed_df.shape[1]} columnas")
print("Columnas resultantes:", preprocessed_df.columns.tolist())
print("\nMuestra del dataset preprocesado:")
print(preprocessed_df.head().to_string(index=False))

premise_tokens = preprocessed_df["premise_processed"].tolist()
print("\nTokens más frecuentes en premise_processed:")
print(preprocessor.top_tokens(premise_tokens, top_n=10).to_string(index=False))
print("\nBigramas más frecuentes en premise_processed:")
print(preprocessor.top_ngrams(premise_tokens, n=2, top_n=10).to_string(index=False))

Filas en inglés: 6870



Dataset preprocesado: 6870 filas x 23 columnas
Columnas resultantes: ['id', 'premise', 'hypothesis', 'lang_abv', 'language', 'label', 'premise_clean', 'premise_tokens', 'premise_tokens_no_stopwords', 'premise_processed', 'premise_2grams', 'premise_processed_2grams', 'premise_3grams', 'premise_processed_3grams', 'hypothesis_clean', 'hypothesis_tokens', 'hypothesis_tokens_no_stopwords', 'hypothesis_processed', 'hypothesis_2grams', 'hypothesis_processed_2grams', 'hypothesis_3grams', 'hypothesis_processed_3grams', 'preprocessing_config']

Muestra del dataset preprocesado:
        id                                                                                      premise                                                                        hypothesis lang_abv language  label                                                                                premise_clean                                                                                              premise_tokens             

In [56]:
preprocessor.export_preprocessed_dataset_csv(df_train, OUTPUT_PATH)
exported_df = pd.read_csv(OUTPUT_PATH)
print(f"Dataset preprocesado exportado a: {OUTPUT_PATH}")
print(f"CSV exportado: {exported_df.shape[0]} filas x {exported_df.shape[1]} columnas")
print("Columnas exportadas:", exported_df.columns.tolist())
print("\nPrimeras filas leídas desde el CSV:")
print(exported_df.head().to_string(index=False))

Dataset preprocesado exportado a: data/train_preprocessed.csv
CSV exportado: 6870 filas x 23 columnas
Columnas exportadas: ['id', 'premise', 'hypothesis', 'lang_abv', 'language', 'label', 'premise_clean', 'premise_tokens', 'premise_tokens_no_stopwords', 'premise_processed', 'premise_2grams', 'premise_processed_2grams', 'premise_3grams', 'premise_processed_3grams', 'hypothesis_clean', 'hypothesis_tokens', 'hypothesis_tokens_no_stopwords', 'hypothesis_processed', 'hypothesis_2grams', 'hypothesis_processed_2grams', 'hypothesis_3grams', 'hypothesis_processed_3grams', 'preprocessing_config']

Primeras filas leídas desde el CSV:
        id                                                                                      premise                                                                        hypothesis lang_abv language  label                                                                                premise_clean                                                                  